In [ ]:
import importlib

from sympy.physics.units import temperature

for pkg in ("torch", "tensorflow", "flax", "transformers"):
    try:
        m = importlib.import_module(pkg)
        print(pkg, "found:", getattr(m, "__version__", "version unknown"))
    except Exception as e:
        print(pkg, "missing:", e)


In [ ]:
response = generator("Write a 500 word story about a boy who gained superpowers by being bitten by a radioactive spider.",
                     max_length=100, # Your Code Here
                     num_return_sequences=1)
print(response)

In [ ]:
import warnings
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1  # 0 = first GPU, -1 = CPU
generator = pipeline('text-generation', model='gpt2', device=device)

output = generator("Hello, my name is", max_new_tokens=50, do_sample=True, top_k=50)
print(output[0]['generated_text'])


In [9]:
# python
import warnings
warnings.filterwarnings("ignore")

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

model_id = "meta-llama/Llama-3.2-1B-Instruct"

# load model onto GPU when available, otherwise CPU
if torch.cuda.is_available():
    device = 0
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
else:
    device = -1
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map={"": "cpu"},
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False, trust_remote_code=True)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device,
)

prompt = "Write a 500 word story about a boy who gained superpowers by being bitten by a radioactive spider."
output = generator(prompt, max_new_tokens=500, do_sample=True, top_k=50, temperature=0.8, num_return_sequences=1)
print(output[0]["generated_text"])


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct.
403 Client Error. (Request ID: Root=1-695d49ca-3896c8155f57da9e51cde97d;eace475c-1852-4d6d-a2ef-c036ada17d07)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct/resolve/main/config.json.
Your request to access model meta-llama/Llama-3.2-1B-Instruct is awaiting a review from the repo authors.

In [10]:
# python
import os
from huggingface_hub import login
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

# 1) Authenticate: set HF_HUB_TOKEN env var or replace the placeholder with your token,
#    or run `huggingface-cli login` in the terminal.
HF_TOKEN = os.getenv("HF_HUB_TOKEN", "<YOUR_HF_TOKEN>")
if HF_TOKEN != "<YOUR_HF_TOKEN>":
    login(token=HF_TOKEN)  # logs into the hub for this session
else:
    print("Warning: set HF_HUB_TOKEN environment variable or run `huggingface-cli login`")

model_id = "meta-llama/Llama-3.2-1B-Instruct"
device = 0 if torch.cuda.is_available() else -1

# 2) Try to load the gated model (will raise 403 if you don't have access).
try:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        trust_remote_code=True,
        use_auth_token=True,
        device_map="auto" if device == 0 else {"": "cpu"},
        torch_dtype=torch.float16 if device == 0 else None,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False, trust_remote_code=True, use_auth_token=True)

    # 3) Create generator BEFORE you use it
    generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=device)

except Exception as e:
    print("Model load failed (likely 403 or missing access):", e)
    print("Ensure you accepted the model license at the model page and your token has access.")
    # fallback to a public small model so notebook continues to work
    generator = pipeline("text-generation", model="gpt2", device=device)

# 4) Use the generator (use max_new_tokens rather than max_length)
prompt = "Write a 500 word story about a boy who gained superpowers by being bitten by a radioactive spider."
output = generator(prompt, max_new_tokens=500, do_sample=True, top_k=50, temperature=0.8, num_return_sequences=1)
print(output[0]["generated_text"])


Model load failed (likely 403 or missing access): Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`
Ensure you accepted the model license at the model page and your token has access.


Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Write a 500 word story about a boy who gained superpowers by being bitten by a radioactive spider.

You can read more about comic books in this series:

The first issue of This American Life begins on Tuesday, April 25th, 2012.


In [13]:
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

generator = pipeline('text-generation', model='meta-llama/Llama-3.2-1B')

Device set to use mps:0


In [19]:
response = generator("Write a 500 word story about a boy who gained superpowers by being bitten by a radioactive spider.",
                     max_length=1024, temperature=0.9, top_k=50, top_p=0.95, # Your Code Here
                     num_return_sequences=1)
print(response)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "Write a 500 word story about a boy who gained superpowers by being bitten by a radioactive spider. Be specific about the details of the bite and the superhero that developed from the superhero. Use your own words to describe the powers of the superhero and the superhero's weaknesses. Make sure you use an external source to support your claims. Make a detailed plot plan. Include a summary and conclusion. Be specific about the superhero's powers and limitations.\nRead the following story about a superhero named Ben. How do you think Ben's powers could be used? Can you think of other ways Ben could use his powers?\nThe Adventures of Ben Franklin (Batman)"}]


In [20]:
response = generator("Write a 500 word story about a boy who gained superpowers by being bitten by a radioactive spider.",
                     max_length=1024,
                     temperature = 0.9,
                     top_k = 50,
                     top_p = 0.95,
                     num_return_sequences=1)
print(response)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Write a 500 word story about a boy who gained superpowers by being bitten by a radioactive spider. Then use this story to create a drawing or illustration. You may draw on your own or use an internet search engine. It is fine to use multiple images in your final product.\nUse this word cloud to brainstorm your story and determine which words are most significant for you.\nYou are to write a story using the 12 words that were important to you. Your story must include each word in your cloud at least once.\nChoose a color to represent your story, then add this to the bottom of your story. Your final project is due by May 10, 2018, in class.'}]
